# 01 — Dataset Exploration & Inspection

**Project:** ROSA Knee AI — AI-Based Knee Bone Segmentation & Surgical Planning  
**Module:** 0 — Medical Imaging Data Foundation  
**Purpose:** Understand exactly what is inside our dataset before doing anything else.

---

## What This Notebook Does

1. **Discovers** all files in the dataset directory
2. **Loads** a single `.npz` file and inspects its contents
3. **Classifies** whether the data is CT intensity or segmentation labels  
   ⚠️ This is the MOST CRITICAL step — we must know what we're working with
4. **Visualizes** slices (axial, coronal, sagittal)
5. **Analyzes** label distribution (if segmentation masks)
6. **Shows** each label separately to understand anatomy
7. **Generates** a 3D preview of the segmentation
8. **Scans** the entire dataset and produces a summary report

### Rules
- We do NOT assume what labels 0–7 represent
- We do NOT modify any raw data files
- We do NOT train any model
- If something is unknown, we print: `UNKNOWN — NEEDS VERIFICATION`

---
## 0. Setup & Imports

In [ ]:
# =============================================================================
# Standard library imports
# =============================================================================
import sys
from pathlib import Path

# =============================================================================
# Scientific computing
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# Add project root to path so we can import our src/ modules
# =============================================================================
# In VS Code: the notebook is in notebooks/, so project root is one level up
# In Google Colab: adjust this path to where you mounted the project
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python path includes project root: {str(PROJECT_ROOT) in sys.path}")

In [ ]:
# =============================================================================
# Import our custom modules
# =============================================================================
from src.io.npz_loader import load_npz, inspect_npz, scan_npz_dataset
from src.visualization.volume_visualizer import (
    show_slice,
    show_three_views,
    show_multi_slice,
    show_label_distribution,
    show_intensity_histogram,
    show_each_label_slice,
    show_3d_preview,
    show_slice_overlay,
)
from src.quality_control.dataset_checker import (
    classify_volume_content,
    scan_dataset,
    generate_dataset_report,
    print_dataset_summary,
)

print("All modules imported successfully!")

In [ ]:
# =============================================================================
# CONFIGURATION — Update these paths for your environment
# =============================================================================

# Path to your raw dataset directory
# For VS Code:  typically ../data/raw relative to the notebook
# For Colab:    update to your Google Drive path, e.g.,
#               Path("/content/drive/MyDrive/ROSA_Knee_AI/data/raw")
DATASET_ROOT = PROJECT_ROOT / "data" / "raw"

# Where to save outputs (reports, figures)
OUTPUT_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = OUTPUT_DIR / "reports"
FIGURES_DIR = OUTPUT_DIR / "figures"

# Create output directories if they don't exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset root:  {DATASET_ROOT}")
print(f"Reports dir:   {REPORTS_DIR}")
print(f"Figures dir:   {FIGURES_DIR}")
print(f"Dataset exists: {DATASET_ROOT.exists()}")

---
## 1. Dataset Discovery

First, let's see what files are in our dataset directory.  
We scan for all supported medical image formats: `.npz`, `.dcm`, `.nii`, `.nii.gz`, `.nrrd`, `.mha`, `.mhd`

In [ ]:
# =============================================================================
# Scan the dataset directory for all medical image files
# =============================================================================
scan_results = scan_dataset(DATASET_ROOT)

# Count total files across all formats
total_files = sum(scan_results['counts'].values())
print(f"\nTotal medical image files found: {total_files}")

if total_files == 0:
    print("\n⚠️  No medical image files found!")
    print(f"Please place your dataset files in: {DATASET_ROOT}")
    print("Supported formats: .npz, .dcm, .nii, .nii.gz, .nrrd, .mha, .mhd")

In [ ]:
# =============================================================================
# List all .npz files (our primary dataset format)
# =============================================================================
npz_files = sorted(Path(f) for f in scan_results['files'].get('.npz', []))

print(f"Number of .npz files: {len(npz_files)}")
print("\nFirst 10 files:")
for i, f in enumerate(npz_files[:10]):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {i+1}. {f.name}  ({size_mb:.1f} MB)")

if len(npz_files) > 10:
    print(f"  ... and {len(npz_files) - 10} more files")

---
## 2. Load and Inspect a Single File

Before doing anything with the dataset, we need to understand what a single file contains.  
We will load ONE `.npz` file and examine:
- What keys are in the file?
- What is the array shape, dtype, value range?
- How many unique values exist?
- **Is this CT intensity data or segmentation labels?**

In [ ]:
# =============================================================================
# Load the FIRST .npz file for inspection
# =============================================================================
if len(npz_files) == 0:
    print("No .npz files found. Please check your DATASET_ROOT path.")
else:
    # Pick the first file for detailed inspection
    sample_file = npz_files[0]
    print(f"Inspecting: {sample_file.name}")
    print(f"Full path:  {sample_file}")
    print(f"File size:  {sample_file.stat().st_size / (1024*1024):.2f} MB")
    print()

In [ ]:
# =============================================================================
# Detailed inspection: load the array, report all properties
# =============================================================================
result = load_npz(sample_file)

# Extract the volume for further analysis
volume = result['volume']

print(f"\n{'='*50}")
print(f"DETAILED ARRAY REPORT")
print(f"{'='*50}")
print(f"Keys in file:      {result['keys']}")
print(f"Shape:             {result['shape']}")
print(f"Dtype:             {result['dtype']}")
print(f"Min value:         {result['min_val']}")
print(f"Max value:         {result['max_val']}")
print(f"Num unique values: {result['num_unique']}")
print(f"Unique values:     {result['unique_values']}")
print(f"Memory usage:      {volume.nbytes / (1024*1024):.1f} MB")
print(f"{'='*50}")

---
## 3. ⚠️ CRITICAL: Classify Data Content

**This is the most important step in the entire notebook.**

We need to determine whether the array `x` in our `.npz` files contains:
- **CT intensity data** (Hounsfield Units, continuous values, typically int16/float32, range: -1024 to +3000)
- **Segmentation labels** (discrete integer values, each number represents a different anatomical structure)

### Why this matters:
- If it's CT data → we can apply windowing, normalization, and feed it to a segmentation model
- If it's segmentation labels → it's already the OUTPUT of segmentation, and we need to find the corresponding CT images
- **Applying CT windowing to labels would destroy the data**
- **Training a segmentation model on labels (instead of CT) would be meaningless**

### Our classification approach:
We use multiple heuristics (dtype, value range, number of unique values, histogram shape)  
but **the final answer must be verified against the dataset documentation.**

In [ ]:
# =============================================================================
# Automated content classification
# =============================================================================
classification = classify_volume_content(volume)

print(f"{'='*60}")
print(f"CONTENT CLASSIFICATION RESULT")
print(f"{'='*60}")
print(f"Classification:  {classification['classification']}")
print(f"Confidence:      {classification['confidence']}")
print(f"Dtype:           {classification['dtype']}")
print(f"Value range:     {classification['value_range']}")
print(f"Unique values:   {classification['num_unique']}")
print(f"\nReasoning:")
for i, reason in enumerate(classification['reasoning'], 1):
    print(f"  {i}. {reason}")
print(f"{'='*60}")

In [ ]:
# =============================================================================
# Manual verification evidence — examine the data yourself
# =============================================================================
print("MANUAL VERIFICATION EVIDENCE")
print("="*50)

# Evidence 1: Dtype
print(f"\n1. Dtype is '{volume.dtype}'")
if volume.dtype == np.uint8:
    print("   → uint8 can store 0-255. CT data is typically int16 (-1024 to +3000 HU).")
    print("   → uint8 is commonly used for segmentation masks.")

# Evidence 2: Unique values
unique_vals = np.unique(volume)
print(f"\n2. Unique values: {unique_vals}")
print(f"   Number of unique values: {len(unique_vals)}")
if len(unique_vals) < 20:
    print("   → Very few discrete values — characteristic of segmentation labels.")
    print("   → CT data would have hundreds or thousands of unique intensity values.")

# Evidence 3: Value distribution
print(f"\n3. Value distribution:")
for val in unique_vals:
    count = np.sum(volume == val)
    pct = (count / volume.size) * 100
    print(f"   Value {val}: {count:>12,} voxels ({pct:>6.2f}%)")

# Evidence 4: Spatial pattern check
print(f"\n4. Gradient check (do neighboring voxels have smooth transitions?):")
mid_z = volume.shape[2] // 2
mid_slice = volume[:, :, mid_z]
# Compute gradient magnitude
grad_y = np.diff(mid_slice.astype(float), axis=0)
grad_x = np.diff(mid_slice.astype(float), axis=1)
# In CT: gradients are smooth with many small values
# In labels: gradients are 0 almost everywhere, with sharp jumps at boundaries
nonzero_grad = np.sum(np.abs(grad_y) > 0)
total_grad = grad_y.size
print(f"   Non-zero gradients: {nonzero_grad:,} / {total_grad:,} ({100*nonzero_grad/total_grad:.2f}%)")
if nonzero_grad / total_grad < 0.1:
    print("   → Very few non-zero gradients — consistent with label masks (flat regions with sharp edges).")
else:
    print("   → Many non-zero gradients — consistent with intensity/CT data (smooth tissue transitions).")

print(f"\n{'='*50}")
print("CONCLUSION: Review the evidence above and verify against dataset documentation.")
print("DO NOT proceed with preprocessing until this is confirmed.")
print(f"{'='*50}")

In [ ]:
# =============================================================================
# Intensity histogram — the DEFINITIVE visual test
# =============================================================================
# CT data: broad, continuous histogram with peaks for air, tissue, bone
# Labels:  a few discrete spikes (one per label value)

print("Intensity/Value Histogram")
print("If you see discrete spikes → segmentation labels")
print("If you see a broad continuous distribution → CT intensity")
print()

show_intensity_histogram(
    volume, 
    bins=max(50, int(result['max_val'] - result['min_val'] + 1)),
    title=f"Value Histogram — {sample_file.name}"
)

---
## 4. Visualize Slices

Now let's look at the data visually. We'll display:
1. **Axial** view (slicing through the Z-axis, top-to-bottom)
2. **Coronal** view (slicing through the Y-axis, front-to-back)
3. **Sagittal** view (slicing through the X-axis, left-to-right)

If the data is segmentation labels, we expect to see colored regions.  
If it's CT data, we expect to see anatomical structures in grayscale.

In [ ]:
# =============================================================================
# Three-plane view (middle slices)
# =============================================================================
print(f"Volume shape: {volume.shape}")
print(f"Axis 0 (Axial/Z):    {volume.shape[0]} slices")
print(f"Axis 1 (Coronal/Y):  {volume.shape[1]} slices")
print(f"Axis 2 (Sagittal/X): {volume.shape[2]} slices")
print()

# Use a colormap that makes labels more visible
# 'tab10' or 'nipy_spectral' for labels; 'gray' for CT
cmap_choice = 'nipy_spectral' if classification['classification'] == 'segmentation_labels' else 'gray'

show_three_views(
    volume,
    title=f"Three-Plane View — {sample_file.name}",
    cmap=cmap_choice
)

In [ ]:
# =============================================================================
# Also show in grayscale for comparison
# This helps us see whether the values look like anatomy or discrete regions
# =============================================================================
show_three_views(
    volume,
    title=f"Three-Plane View (grayscale) — {sample_file.name}",
    cmap='gray'
)

In [ ]:
# =============================================================================
# Multi-slice view — 9 evenly spaced slices through the axial axis
# This gives a quick overview of the entire volume
# =============================================================================
show_multi_slice(
    volume, 
    axis=0,  # Axial
    num_slices=9,
    cmap=cmap_choice
)

In [ ]:
# =============================================================================
# Interactive: view a specific slice
# Change AXIS and SLICE_INDEX to explore different parts of the volume
# =============================================================================
AXIS = 0            # 0=axial, 1=coronal, 2=sagittal
SLICE_INDEX = None   # None = middle slice; or set to a specific number

show_slice(
    volume, 
    axis=AXIS, 
    index=SLICE_INDEX,
    title=f"{sample_file.name}",
    cmap=cmap_choice
)

---
## 5. Label Analysis (if segmentation masks)

If the classification says this is segmentation labels, let's analyze:  
- How many voxels belong to each label?
- What percentage of the volume does each label occupy?
- What does each label look like in isolation?

**⚠️ We do NOT assume what the labels mean.**  
Label 1 could be femur, tibia, patella, or something entirely different.  
This must be verified against the dataset documentation.

In [ ]:
# =============================================================================
# Label distribution — bar chart + table
# =============================================================================
print(f"Classification: {classification['classification']}")
print(f"Unique labels in volume: {np.unique(volume)}")
print()

show_label_distribution(
    volume, 
    title=f"Label Distribution — {sample_file.name}"
)

In [ ]:
# =============================================================================
# Show each label in isolation at the middle axial slice
# This helps us see the spatial extent and shape of each label
# =============================================================================
# Find a slice that contains multiple labels (not just background)
# We check several slices and pick the one with the most labels
best_slice_idx = 0
best_num_labels = 0
for z in range(0, volume.shape[0], max(1, volume.shape[0] // 20)):
    num_labels = len(np.unique(volume[z, :, :])) - 1  # exclude background (0)
    if num_labels > best_num_labels:
        best_num_labels = num_labels
        best_slice_idx = z

print(f"Best slice for label visualization: {best_slice_idx} (contains {best_num_labels} non-background labels)")
print(f"Labels present in this slice: {np.unique(volume[best_slice_idx, :, :])}")
print()

show_each_label_slice(
    volume, 
    axis=0, 
    index=best_slice_idx
)

In [ ]:
# =============================================================================
# Overlay view — show labels colored on top of the volume
# Since we may only have labels (no separate CT image), we overlay labels on themselves
# The "image" channel is the volume displayed in grayscale
# The "mask" channel is the same volume displayed as colored labels
# =============================================================================
show_slice_overlay(
    image=volume,
    mask=volume,
    axis=0,
    index=best_slice_idx,
    alpha=0.5
)

In [ ]:
# =============================================================================
# Show isolated labels across all three planes
# =============================================================================
print("Sagittal view — each label isolated:")
show_each_label_slice(volume, axis=2, index=None)

print("\nCoronal view — each label isolated:")
show_each_label_slice(volume, axis=1, index=None)

---
## 6. 3D Preview

If this is a segmentation mask, we can render a basic 3D surface for each label  
to verify the masks form coherent anatomical structures.

**This is NOT production-quality 3D rendering.**  
It's a quick sanity check using marching cubes + matplotlib.

Note: This may take 10–30 seconds depending on volume size.

In [ ]:
# =============================================================================
# 3D surface preview of segmentation labels
# =============================================================================
print("Generating 3D preview (this may take a moment)...")
print("Note: The volume will be downsampled for faster rendering.")
print()

# Render all non-background labels
labels_to_render = [int(l) for l in np.unique(volume) if l > 0]
print(f"Labels to render: {labels_to_render}")

show_3d_preview(
    volume,
    labels_to_show=labels_to_render,
    spacing=(1.0, 1.0, 1.0),  # We don't have spacing info from .npz
    max_size=100,             # Downsample to max 100 voxels per dimension
    figsize=(12, 12)
)

---
## 7. Inspect Additional Cases

Let's check a few more files to verify consistency across the dataset.  
Are all files the same shape? Same dtype? Same label set?

In [ ]:
# =============================================================================
# Quick inspection of multiple files (up to 5)
# =============================================================================
num_to_check = min(5, len(npz_files))
print(f"Inspecting {num_to_check} files for consistency...\n")

consistency_data = []

for i, f in enumerate(npz_files[:num_to_check]):
    print(f"--- File {i+1}/{num_to_check}: {f.name} ---")
    info = inspect_npz(f)
    
    # Extract metadata for the main array key
    for key in info.get('keys', []):
        key_info = info.get(key, {})
        consistency_data.append({
            'file': f.name,
            'key': key,
            'shape': str(key_info.get('shape', 'N/A')),
            'dtype': key_info.get('dtype', 'N/A'),
            'min': key_info.get('min_val', 'N/A'),
            'max': key_info.get('max_val', 'N/A'),
            'num_unique': key_info.get('num_unique', 'N/A'),
            'content_type': key_info.get('estimated_content_type', 'N/A'),
        })
    print()

# Display as a table
if consistency_data:
    consistency_df = pd.DataFrame(consistency_data)
    print("\nConsistency Check Summary:")
    display(consistency_df) if 'display' in dir() else print(consistency_df.to_string())

In [ ]:
# =============================================================================
# Check: Do all files have the same shape?
# =============================================================================
if consistency_data:
    shapes = [d['shape'] for d in consistency_data]
    unique_shapes = set(shapes)
    
    print(f"Unique shapes found: {len(unique_shapes)}")
    for s in unique_shapes:
        count = shapes.count(s)
        print(f"  {s}: {count} files")
    
    if len(unique_shapes) == 1:
        print("\n✅ All files have the same shape — good!")
    else:
        print("\n⚠️ Files have DIFFERENT shapes — this may require special handling.")

---
## 8. Full Dataset Scan & Report

Now let's scan ALL files in the dataset and generate a comprehensive CSV report.  
This may take a few minutes for large datasets.

The report will be saved to: `outputs/reports/dataset_report.csv`

In [ ]:
# =============================================================================
# Generate full dataset report
# =============================================================================
report_path = REPORTS_DIR / "dataset_report.csv"

print(f"Generating dataset report...")
print(f"Scanning: {DATASET_ROOT}")
print(f"Report will be saved to: {report_path}")
print()

report_df = generate_dataset_report(DATASET_ROOT, output_path=report_path)

In [ ]:
# =============================================================================
# Display the report
# =============================================================================
if not report_df.empty:
    print(f"Report contains {len(report_df)} entries.")
    print("\nFirst 10 rows:")
    display(report_df.head(10)) if 'display' in dir() else print(report_df.head(10).to_string())
else:
    print("Report is empty — no files were successfully processed.")

In [ ]:
# =============================================================================
# Print formatted summary
# =============================================================================
print_dataset_summary(report_df)

---
## 9. Dataset Summary

This is the final summary of everything we've learned about the dataset.

In [ ]:
# =============================================================================
# FINAL DATASET SUMMARY
# =============================================================================
print("="*60)
print("          DATASET SUMMARY")
print("="*60)
print()

if not report_df.empty:
    n_cases = len(report_df)
    n_ok = len(report_df[report_df['status'] == 'OK'])
    n_err = len(report_df[report_df['status'] == 'ERROR'])
    n_warn = len(report_df[report_df['status'] == 'WARNING'])
    
    # Content type breakdown
    content_types = report_df['content_type'].value_counts()
    
    # Shape info
    shapes = report_df['shape'].dropna().astype(str)
    
    print(f"Number of files:          {n_cases}")
    print(f"Healthy files (OK):       {n_ok}")
    print(f"Files with errors:        {n_err}")
    print(f"Files with warnings:      {n_warn}")
    print()
    print(f"Content type breakdown:")
    for ctype, count in content_types.items():
        print(f"  {ctype}: {count}")
    print()
    print(f"Unique shapes:            {shapes.nunique()}")
    print(f"Most common shape:        {shapes.mode().iloc[0] if len(shapes) > 0 else 'N/A'}")
    print()
    
    # Check for segmentation-specific info
    seg_files = report_df[report_df['content_type'] == 'segmentation_labels']
    if len(seg_files) > 0:
        print(f"Segmentation masks found: {len(seg_files)}")
        print(f"Label range:              0 to {int(report_df['max_val'].max())}")
        print(f"Label meanings:           UNKNOWN — NEEDS VERIFICATION")
        print(f"                          Do NOT assume label-to-anatomy mapping.")
        print(f"                          Check dataset documentation.")
    print()
    
    # Missing data check
    print(f"Missing CT images:        UNKNOWN — NEEDS VERIFICATION")
    print(f"                          If .npz contains only labels, we need")
    print(f"                          to find the corresponding CT source data.")
    print()
    print(f"Spatial metadata:         NOT AVAILABLE in .npz format")
    print(f"                          .npz files do not store spacing, origin,")
    print(f"                          or direction information.")
    print(f"                          This must come from the original DICOM/NIfTI files.")

else:
    print("No data found. Check DATASET_ROOT path.")

print()
print("="*60)
print("Report saved to:", report_path)
print("="*60)

---
## 10. Verification Checklist

Before moving to the next module, confirm the following:

| # | Check | Status |
|---|-------|--------|
| 1 | Dataset discovered — file types and counts reported | ☐ |
| 2 | `.npz` file loaded — array key, shape, dtype confirmed | ☐ |
| 3 | Content classified — CT intensity vs segmentation labels | ☐ |
| 4 | Axial view displayed correctly | ☐ |
| 5 | Coronal view displayed correctly | ☐ |
| 6 | Sagittal view displayed correctly | ☐ |
| 7 | Label distribution analyzed | ☐ |
| 8 | Individual labels visualized | ☐ |
| 9 | 3D preview generated | ☐ |
| 10 | Multiple files checked for consistency | ☐ |
| 11 | Full dataset report generated (CSV) | ☐ |
| 12 | No raw files modified | ☐ |
| 13 | Label meanings NOT assumed | ☐ |

### Next Steps (after verification):
1. **Verify label meanings** with dataset documentation
2. **Find CT source data** if `.npz` files contain only segmentation labels
3. **Determine spacing/origin** from original medical image files
4. **Move to preprocessing** (notebook `02_preprocessing_demo.ipynb`)